In [4]:
import json
import re
import csv
from pathlib import Path

In [2]:
ARK_PATTERN = re.compile(
    r"ark:/77981/gmgs[0-9bcdfghjkmnpqrstvwxz]{7}"
)

def build_noid_entry(row: dict[str, str]) -> dict:
    """Build one validated, non-mutating NOID reconstruction entry."""
    arks = [
        value.strip()
        for value in row["Identifier"].split("|")
        if ARK_PATTERN.fullmatch(value.strip())
    ]

    if len(arks) != 1:
        raise ValueError(f"Expected one canonical ARK; found {len(arks)}")

    ark = arks[0]
    aardvark_id = row["ID"].strip()
    expected_id = ark.replace("ark:/77981/", "ark:-77981-", 1)

    if aardvark_id != expected_id:
        raise ValueError(
            f"ID mismatch: expected {expected_id!r}, got {aardvark_id!r}"
        )

    references = json.loads(row["dct_references_s"])
    where = references.get("http://schema.org/url")

    if not where:
        raise ValueError("Missing http://schema.org/url reference")

    expected_where = f"https://geodiscovery.uwm.edu/catalog/{aardvark_id}"
    if where != expected_where:
        raise ValueError(
            f"Resolver mismatch: expected {expected_where!r}, got {where!r}"
        )

    bindings = {
        "identifier": ark,
        "ogm_aardvark_id": aardvark_id,
        "title": row["Title"].strip(),
        "access": row["Access Rights"].strip(),
        "where": where,
    }

    download = references.get("http://schema.org/downloadUrl")
    if download:
        bindings["download"] = download

    return {
        "noid": ark.removeprefix("ark:/"),
        "hold": True,
        "bindings": bindings,
    }

In [5]:
csv_path = Path("../csv2JSON/aardvark_export_2026-09-14.csv")

with csv_path.open(encoding="utf-8-sig", newline="") as file:
    first_row = next(csv.DictReader(file))

entry = build_noid_entry(first_row)
entry

{'noid': '77981/gmgs0c4sj74',
 'hold': True,
 'bindings': {'identifier': 'ark:/77981/gmgs0c4sj74',
  'ogm_aardvark_id': 'ark:-77981-gmgs0c4sj74',
  'title': 'Parcels Milwaukee, Wisconsin April 7, 2014',
  'access': 'Public',
  'where': 'https://geodiscovery.uwm.edu/catalog/ark:-77981-gmgs0c4sj74',
  'download': 'https://geodata.uwm.edu/public/gmgs0c4sj74/Milwaukee_Parcels_20140407.zip'}}

In [6]:
assert entry["noid"] == "77981/gmgs0c4sj74"
assert entry["hold"] is True
assert entry["bindings"]["identifier"] == "ark:/77981/gmgs0c4sj74"
assert entry["bindings"]["ogm_aardvark_id"] == "ark:-77981-gmgs0c4sj74"
assert entry["bindings"]["where"].endswith("ark:-77981-gmgs0c4sj74")
assert entry["bindings"]["access"] == "Public"

print("Entry validation passed.")

Entry validation passed.


In [7]:
entries = []
errors = []

with csv_path.open(encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    for row_number, row in enumerate(reader, start=2):
        try:
            entries.append(build_noid_entry(row))
        except (KeyError, TypeError, ValueError, json.JSONDecodeError) as error:
            errors.append(
                {
                    "row": row_number,
                    "id": row.get("ID", ""),
                    "error": str(error),
                }
            )

print(f"Valid entries: {len(entries)}")
print(f"Errors: {len(errors)}")

errors[:10]

Valid entries: 930
Errors: 0


[]

In [8]:
noids = [entry["noid"] for entry in entries]
where_urls = [entry["bindings"]["where"] for entry in entries]

assert len(entries) == 930
assert len(noids) == len(set(noids)), "Duplicate NOIDs found"
assert len(where_urls) == len(set(where_urls)), "Duplicate resolver URLs found"
assert all(entry["hold"] is True for entry in entries)
assert all(entry["bindings"]["where"].startswith(
    "https://geodiscovery.uwm.edu/catalog/ark:-77981-gmgs"
) for entry in entries)

with_download = sum(
    "download" in entry["bindings"]
    for entry in entries
)

print("Manifest invariants passed.")
print(f"Hold and bind: {len(entries)}")
print(f"With download binding: {with_download}")
print(f"Without download binding: {len(entries) - with_download}")

Manifest invariants passed.
Hold and bind: 930
With download binding: 911
Without download binding: 19


In [9]:
live_only_noids = {
    "77981/gmgs15dv46t",
    "77981/gmgs2rbp036",
    "77981/gmgs3xsj48c",
    "77981/gmgs4xgxdk4",
    "77981/gmgs7pvmdn2",
    "77981/gmgscc2fs30",
    "77981/gmgsh44j2g5",
    "77981/gmgsmw6mbvg",
    "77981/gmgsrn8pn7p",
}

exported_noids = {entry["noid"] for entry in entries}

assert live_only_noids.isdisjoint(exported_noids)

hold_noids = sorted(exported_noids | live_only_noids)

assert len(hold_noids) == 939
assert len(set(hold_noids)) == 939

print(f"Total holds: {len(hold_noids)}")
print(f"Bound records: {len(entries)}")
print(f"Hold-only records: {len(live_only_noids)}")

Total holds: 939
Bound records: 930
Hold-only records: 9


In [10]:
import os
import subprocess

noid_bin = Path("perl5/bin/noid").resolve()
db_dir = Path("scratch/gmgs").resolve()
perl5lib = str(Path("perl5/lib/perl5").resolve())

result = subprocess.run(
    [
        str(noid_bin),
        "-f",
        str(db_dir),
        "validate",
        "-",
        *hold_noids,
    ],
    env={**os.environ, "PERL5LIB": perl5lib},
    capture_output=True,
    text=True,
    check=True,
)

validation_lines = result.stdout.splitlines()
validation_errors = [
    line for line in validation_lines
    if line.startswith("iderr:")
]

assert len(validation_lines) == 939
assert not validation_errors, validation_errors[:10]

print("All 939 identifiers passed native NOID validation.")

All 939 identifiers passed native NOID validation.


In [11]:
repeat_hold = subprocess.run(
    [
        str(noid_bin),
        "-f",
        str(db_dir),
        "hold",
        "set",
        "77981/gmgsh41jm0c",
    ],
    env={**os.environ, "PERL5LIB": perl5lib},
    capture_output=True,
    text=True,
)

print("Return code:", repeat_hold.returncode)
print("stdout:", repeat_hold.stdout.strip())
print("stderr:", repeat_hold.stderr.strip())

Return code: 0
stdout: ok: 1 hold placed
stderr: 


In [12]:
hold_result = subprocess.run(
    [
        str(noid_bin),
        "-f",
        str(db_dir),
        "hold",
        "set",
        *hold_noids,
    ],
    env={**os.environ, "PERL5LIB": perl5lib},
    capture_output=True,
    text=True,
    check=True,
)

print(hold_result.stdout.strip())

ok: 939 holds placed


In [13]:
dbinfo_result = subprocess.run(
    [str(noid_bin), "-f", str(db_dir), "dbinfo"],
    env={**os.environ, "PERL5LIB": perl5lib},
    capture_output=True,
    text=True,
    check=True,
)

held_match = re.search(r"^:/held:\s+(\d+)$", dbinfo_result.stdout, re.MULTILINE)
assert held_match, "NOID dbinfo did not report a held count"
assert int(held_match.group(1)) == 939

print("Scratch database confirms 939 holds.")

AssertionError: NOID dbinfo did not report a held count

In [14]:
[
    repr(line)
    for line in dbinfo_result.stdout.splitlines()
    if "/held:" in line
]

["'  :/held: 941'"]

In [15]:
dump_result = subprocess.run(
    [str(noid_bin), "-f", str(db_dir), "dbinfo", "dump"],
    env={**os.environ, "PERL5LIB": perl5lib},
    capture_output=True,
    text=True,
    check=True,
)

held_noids_in_db = {
    line.split("\t", 1)[0].strip()
    for line in dump_result.stdout.splitlines()
    if "\t:/h:" in line
}

assert held_noids_in_db == set(hold_noids)

print(f"Unique hold keys confirmed: {len(held_noids_in_db)}")
print("Administrative held counter:", 941)

Unique hold keys confirmed: 939
Administrative held counter: 941


In [16]:
missing_holds = sorted(set(hold_noids) - held_noids_in_db)

print(f"Requested holds: {len(hold_noids)}")
print(f"Existing holds: {len(held_noids_in_db)}")
print(f"New holds required: {len(missing_holds)}")

assert missing_holds == []

Requested holds: 939
Existing holds: 939
New holds required: 0


In [17]:
sample_entry = next(
    entry
    for entry in entries
    if entry["noid"] == "77981/gmgsh41jm0c"
)

binding_results = {}

for element, value in sample_entry["bindings"].items():
    result = subprocess.run(
        [
            str(noid_bin),
            "-f",
            str(db_dir),
            "bind",
            "set",
            sample_entry["noid"],
            element,
            value,
        ],
        env={**os.environ, "PERL5LIB": perl5lib},
        capture_output=True,
        text=True,
        check=True,
    )
    binding_results[element] = result.stdout.strip()

print(f"Bindings written: {len(binding_results)}")
for element, result in binding_results.items():
    print(f"{element}: {result.splitlines()[-1]}")

Bindings written: 6
identifier: Status:  ok, 22 bytes written, replacing 0 bytes
ogm_aardvark_id: Status:  ok, 22 bytes written, replacing 0 bytes
title: Status:  ok, 39 bytes written, replacing 0 bytes
access: Status:  ok, 10 bytes written, replacing 0 bytes
where: Status:  ok, 59 bytes written, replacing 59 bytes
download: Status:  ok, 88 bytes written, replacing 0 bytes


In [18]:
binding_mismatches = {}

for element, expected_value in sample_entry["bindings"].items():
    result = subprocess.run(
        [
            str(noid_bin),
            "-f",
            str(db_dir),
            "get",
            sample_entry["noid"],
            element,
        ],
        env={**os.environ, "PERL5LIB": perl5lib},
        capture_output=True,
        text=True,
        check=True,
    )

    actual_value = result.stdout.rstrip("\r\n")

    if actual_value != expected_value:
        binding_mismatches[element] = {
            "expected": expected_value,
            "actual": actual_value,
        }

assert not binding_mismatches, binding_mismatches
print("All sample bindings verified exactly.")

All sample bindings verified exactly.


In [19]:
transport_issues = []

for entry in entries:
    for element, value in entry["bindings"].items():
        if not isinstance(value, str):
            transport_issues.append(
                (entry["noid"], element, "not a string")
            )
        elif "\x00" in value:
            transport_issues.append(
                (entry["noid"], element, "contains a null byte")
            )
        elif "\n" in value or "\r" in value:
            transport_issues.append(
                (entry["noid"], element, "contains a newline")
            )

print(f"Transport issues: {len(transport_issues)}")
transport_issues[:10]

Transport issues: 0


[]

In [20]:
operations = [
    {
        "operation": "hold",
        "action": "set",
        "noid": noid,
    }
    for noid in hold_noids
]

for entry in sorted(entries, key=lambda item: item["noid"]):
    for element, value in entry["bindings"].items():
        operations.append(
            {
                "operation": "bind",
                "action": "set",
                "noid": entry["noid"],
                "element": element,
                "value": value,
            }
        )

hold_operations = sum(
    operation["operation"] == "hold"
    for operation in operations
)
bind_operations = sum(
    operation["operation"] == "bind"
    for operation in operations
)

assert hold_operations == 939
assert bind_operations == 5561

print(f"Total planned operations: {len(operations)}")
print(f"Hold operations: {hold_operations}")
print(f"Bind operations: {bind_operations}")

Total planned operations: 6500
Hold operations: 939
Bind operations: 5561


In [21]:
import hashlib

manifest_text = json.dumps(
    operations,
    ensure_ascii=False,
    indent=2,
    sort_keys=True,
) + "\n"

manifest_sha256 = hashlib.sha256(
    manifest_text.encode("utf-8")
).hexdigest()

print(f"Manifest records: {len(operations)}")
print(f"Manifest bytes: {len(manifest_text.encode('utf-8'))}")
print(f"SHA-256: {manifest_sha256}")

Manifest records: 6500
Manifest bytes: 1026074
SHA-256: 4c92ed38965690f67f0aac7fbda2f508a6528e70add4db311eb7252cfbca8a9c


In [22]:
manifest_path = Path("scratch/rebuild-manifest.json")
manifest_path.write_text(manifest_text, encoding="utf-8")

checksum_path = manifest_path.with_suffix(".json.sha256")
checksum_path.write_text(
    f"{manifest_sha256}  {manifest_path.name}\n",
    encoding="utf-8",
)

assert hashlib.sha256(manifest_path.read_bytes()).hexdigest() == manifest_sha256

print(f"Wrote: {manifest_path}")
print(f"Wrote: {checksum_path}")

Wrote: scratch/rebuild-manifest.json
Wrote: scratch/rebuild-manifest.json.sha256


In [23]:
rebuild_db_dir = Path("scratch/rebuild-clean").resolve()
assert not (rebuild_db_dir / "NOID").exists(), "Clean rebuild database already exists"

rebuild_db_dir.mkdir(parents=True, exist_ok=True)

create_result = subprocess.run(
    [
        str(noid_bin),
        "-f",
        str(rebuild_db_dir),
        "dbcreate",
        "gmgs.reeeeeek",
        "long",
        "77981",
        "University of Wisconsin-Milwaukee Libraries",
        "gmgs",
    ],
    env={**os.environ, "PERL5LIB": perl5lib},
    capture_output=True,
    text=True,
    check=True,
)

print(create_result.stdout.strip())

Created:   minter for 594823321 random identifiers of form gmgs.reeeeeek
       A Noid minting and binding database has been created that will bind
       and mint 594,823,321 identifiers with the template "gmgs.reeeeeek".
       Sample identifiers would be "77981/gmgs7zrzszh" and "77981/gmgs022sr7m".
       Minting order is random.  See /home/srappel/snakepit/notebooks/noid_wrapper/scratch/rebuild-clean/NOID/README for details.


In [24]:
clean_dbinfo = subprocess.run(
    [str(noid_bin), "-f", str(rebuild_db_dir), "dbinfo"],
    env={**os.environ, "PERL5LIB": perl5lib},
    capture_output=True,
    text=True,
    check=True,
).stdout

def dbinfo_number(name: str) -> int:
    match = re.search(
        rf"^\s*:/{re.escape(name)}:\s+(\d+)\s*$",
        clean_dbinfo,
        re.MULTILINE,
    )
    if not match:
        raise ValueError(f"Missing dbinfo value: {name}")
    return int(match.group(1))

assert dbinfo_number("oacounter") == 0
assert dbinfo_number("held") == 0

print("Clean rebuild database confirmed.")

Clean rebuild database confirmed.


In [25]:
hold_result = subprocess.run(
    [
        str(noid_bin),
        "-f",
        str(rebuild_db_dir),
        "hold",
        "set",
        *hold_noids,
    ],
    env={**os.environ, "PERL5LIB": perl5lib},
    capture_output=True,
    text=True,
    check=True,
)

rebuild_dump = subprocess.run(
    [str(noid_bin), "-f", str(rebuild_db_dir), "dbinfo", "dump"],
    env={**os.environ, "PERL5LIB": perl5lib},
    capture_output=True,
    text=True,
    check=True,
).stdout

actual_holds = {
    line.split("\t", 1)[0].strip()
    for line in rebuild_dump.splitlines()
    if "\t:/h:" in line
}

assert actual_holds == set(hold_noids)

held_counter = re.search(
    r"^\s*:/held:\s+(\d+)\s*$",
    rebuild_dump,
    re.MULTILINE,
)
assert held_counter
assert int(held_counter.group(1)) == 939

print("Clean database contains exactly 939 holds.")

Clean database contains exactly 939 holds.


In [26]:
binding_input = "".join(
    f"{element}: {value}\n"
    for element, value in sample_entry["bindings"].items()
) + "\n"

batch_test = subprocess.run(
    [
        str(noid_bin),
        "-f",
        str(db_dir),  # experimental database
        "bind",
        "set",
        sample_entry["noid"],
        ":",
    ],
    env={**os.environ, "PERL5LIB": perl5lib},
    input=binding_input,
    capture_output=True,
    text=True,
)

print("Return code:", batch_test.returncode)
print(batch_test.stdout)
print(batch_test.stderr)


Return code: 0
Id:      77981/gmgsh41jm0c
Element: identifier
Bind:    set
Status:  ok, 22 bytes written, replacing 22 bytes

Id:      77981/gmgsh41jm0c
Element: ogm_aardvark_id
Bind:    set
Status:  ok, 22 bytes written, replacing 22 bytes

Id:      77981/gmgsh41jm0c
Element: title
Bind:    set
Status:  ok, 39 bytes written, replacing 39 bytes

Id:      77981/gmgsh41jm0c
Element: access
Bind:    set
Status:  ok, 10 bytes written, replacing 10 bytes

Id:      77981/gmgsh41jm0c
Element: where
Bind:    set
Status:  ok, 59 bytes written, replacing 59 bytes

Id:      77981/gmgsh41jm0c
Element: download
Bind:    set
Status:  ok, 88 bytes written, replacing 88 bytes





In [27]:
def bind_noid_entry(entry: dict, database: Path) -> None:
    """Bind one validated manifest entry using one NOID invocation."""
    payload = "".join(
        f"{element}: {value}\n"
        for element, value in entry["bindings"].items()
    ) + "\n"

    result = subprocess.run(
        [
            str(noid_bin),
            "-f",
            str(database),
            "bind",
            "set",
            entry["noid"],
            ":",
        ],
        env={**os.environ, "PERL5LIB": perl5lib},
        input=payload,
        capture_output=True,
        text=True,
    )

    expected = len(entry["bindings"])
    successful = result.stdout.count("Status:  ok")

    if result.returncode != 0 or successful != expected:
        raise RuntimeError(
            f"Binding failed for {entry['noid']}: "
            f"{successful}/{expected} bindings succeeded\n"
            f"{result.stderr}"
        )

In [28]:
bound_count = 0

for entry in sorted(entries, key=lambda item: item["noid"]):
    bind_noid_entry(entry, rebuild_db_dir)
    bound_count += 1

    if bound_count % 100 == 0:
        print(f"Bound {bound_count} records")

assert bound_count == 930
print("Bound all 930 records.")

Bound 100 records
Bound 200 records
Bound 300 records
Bound 400 records
Bound 500 records
Bound 600 records
Bound 700 records
Bound 800 records
Bound 900 records
Bound all 930 records.


In [29]:
verification_dump = subprocess.run(
    [str(noid_bin), "-f", str(rebuild_db_dir), "dbinfo", "dump"],
    env={**os.environ, "PERL5LIB": perl5lib},
    capture_output=True,
    text=True,
    check=True,
).stdout

actual_bindings = {}

for line in verification_dump.splitlines():
    if "\t" not in line:
        continue

    noid, stored = line.split("\t", 1)

    if not noid.startswith("77981/gmgs") or stored.startswith(":/"):
        continue

    element, separator, value = stored.partition(": ")
    if separator:
        actual_bindings[(noid, element)] = value

expected_bindings = {
    (entry["noid"], element): value
    for entry in entries
    for element, value in entry["bindings"].items()
}

missing = expected_bindings.keys() - actual_bindings.keys()
unexpected = actual_bindings.keys() - expected_bindings.keys()
mismatched = {
    key: {
        "expected": expected_bindings[key],
        "actual": actual_bindings[key],
    }
    for key in expected_bindings.keys() & actual_bindings.keys()
    if expected_bindings[key] != actual_bindings[key]
}

assert not missing, list(missing)[:10]
assert not unexpected, list(unexpected)[:10]
assert not mismatched, list(mismatched.items())[:10]
assert len(actual_bindings) == 5561

print("Verified all 5,561 bindings exactly.")

Verified all 5,561 bindings exactly.


In [30]:
final_dbinfo = subprocess.run(
    [str(noid_bin), "-f", str(rebuild_db_dir), "dbinfo"],
    env={**os.environ, "PERL5LIB": perl5lib},
    capture_output=True,
    text=True,
    check=True,
).stdout

def final_dbinfo_number(name: str) -> int:
    match = re.search(
        rf"^\s*:/{re.escape(name)}:\s+(\d+)\s*$",
        final_dbinfo,
        re.MULTILINE,
    )
    if not match:
        raise ValueError(f"Missing dbinfo value: {name}")
    return int(match.group(1))

assert final_dbinfo_number("held") == 939
assert final_dbinfo_number("oacounter") == 0
assert actual_holds == set(hold_noids)
assert len(actual_bindings) == 5561
assert not any(
    noid in live_only_noids
    for noid, element in actual_bindings
)

print("Final scratch state:")
print("  Minted: 0")
print("  Unique holds: 939")
print("  Bound records: 930")
print("  Bindings: 5,561")
print("  Hold-only records: 9")

Final scratch state:
  Minted: 0
  Unique holds: 939
  Bound records: 930
  Bindings: 5,561
  Hold-only records: 9
